In [1]:
import os
from dotenv import load_dotenv
from google.cloud import bigquery
import pandas as pd
import numpy as np
import plotly.express as px
from  plotly.subplots import make_subplots
import plotly.graph_objects as go


In [8]:
load_dotenv()

key_file_path = os.environ.get("GCP_KEY_PATH")

client = bigquery.Client.from_service_account_json(
    key_file_path,
    project="quantum-echo-data-eng-prod"
)


query_sales = """
    SELECT 
        product_key, 
        customer_key,
        order_date, 
        order_number,
        quantity,
        gross_sales_amount, 
        unit_price 
    FROM `quantum-echo-data-eng-prod.gold.fct_sales`
"""

query_products = """
     SELECT 
        product_key,
        product_name, 
        category 
    FROM `quantum-echo-data-eng-prod.gold.dim_products`
"""

query_customers = """
     SELECT 
        customer_key,
        first_name, 
        last_name,
        country
    FROM `quantum-echo-data-eng-prod.gold.dim_customers`
"""

df_sales = client.query(query_sales).to_dataframe()
df_products = client.query(query_products).to_dataframe()
df_customers = client.query(query_customers).to_dataframe()

df_sales['order_date'] = pd.to_datetime(df_sales['order_date'])

# 1. Merge all three tables
df_merged = (
    df_sales
    .merge(df_products, on="product_key", how="left")
    .merge(df_customers, on="customer_key", how="left")  
)

# 2. Compute KPIs into a clean 1D Series
kpis = pd.Series({
    "total_sales":  df_merged["gross_sales_amount"].sum(),
    "total_orders": df_merged["order_number"].nunique(),
    "total_quantity": df_merged["quantity"].sum(),
    "total_customers": df_merged["customer_key"].nunique()
})

# 3. Extract and display values
print(f"Total Sales:     ${kpis['total_sales']:,.2f}")
print(f"Total Orders:    {int(kpis['total_orders']):,}")
print(f"Total Quantity:  {int(kpis['total_quantity']):,}")
print(f"Total Customers: {int(kpis['total_customers']):,}")


Total Sales:     $29,355,866.00
Total Orders:    27,659
Total Quantity:  60,423
Total Customers: 18,484


In [9]:
df_customers

,customer_key,first_name,last_name,country
0,ae224a2b6d1b233a204dad19a6aa017d,Aaron,Campbell,NaN
1,fbff791ef0770855e599ea6f87d41653,Aaron,Jenkins,NaN
2,d0881bf5a7895640b88372a784a7eec4,Aaron,Butler,NaN
3,dd1f2ad3a1bf536bc3d7ea8628152941,Abigail,Patterson,NaN
4,9063366e3ba3efafeddfb78e4e422c62,Abigail,Reed,NaN
...,...,...,...,...
18479,9301cb784fa8d1f29d1125c71184ab94,Xavier,Hayes,United States
18480,2ed82a7e645e45584d3aabab834eef35,Xavier,Campbell,United States
18481,842f8aa34c56102c83aaed81f3b175d8,Zachary,Martin,United States
18482,040cea6d24ed05b83f0db871f6794b61,Zoe,Rogers,United States
